In [ ]:
#!/usr/bin/env python3
# pip install seqeval
import os
import time
import asyncio
import aiohttp
import pandas as pd
from google.colab import drive
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
import nest_asyncio

nest_asyncio.apply()
drive.mount('/content/drive')

API_KEY         = ""
OPENROUTER_URL  = "https://openrouter.ai/api/v1/chat/completions"
CSV_PATH        = "sampled_2000.csv"
MODEL           = "anthropic/claude-haiku-4.5"
OUTPUT_DIR      = "/content/drive/MyDrive/bangla_ner_benchmark"
NUM_SAMPLES     = None
CONCURRENCY     = 5
BATCH_SIZE      = 15
BATCH_DELAY     = 5
SITE_URL        = "https://colab.research.google.com"
SITE_NAME       = "Bangla NER Benchmark"
MAX_RETRIES     = 1
BASE_BACKOFF    = 2.0
REQUEST_TIMEOUT = 90

# Ground truth uses these (with spaces)
GT_LABEL_MAP = {
    "B-Health":   "B-Health_Condition",
    "I-Health":   "I-Health_Condition",
    "B-Medical":  "B-Medical_Procedure",
    "I-Medical":  "I-Medical_Procedure",
}

# All valid labels use underscore (no spaces) to avoid parsing issues
VALID_LABELS = [
    "O",
    "B-Symptom",            "I-Symptom",
    "B-Health_Condition",   "I-Health_Condition",
    "B-Medicine",           "I-Medicine",
    "B-Specialist",         "I-Specialist",
    "B-Age",                "I-Age",
    "B-Dosage",             "I-Dosage",
    "B-Medical_Procedure",  "I-Medical_Procedure",
]
VALID_SET   = set(VALID_LABELS)
VALID_LOWER = {v.lower(): v for v in VALID_LABELS}

SYSTEM_PROMPT = """You are a Bangla Biomedical Named Entity Recognition (BioNER) model.

Label each input token using the IOB2 scheme.

Entity types and their tags:
- Symptom          → B-Symptom, I-Symptom
- Health Condition → B-Health_Condition, I-Health_Condition
- Medicine         → B-Medicine, I-Medicine
- Dosage           → B-Dosage, I-Dosage
- Medical Procedure→ B-Medical_Procedure, I-Medical_Procedure
- Specialist       → B-Specialist, I-Specialist
- Age              → B-Age, I-Age
- Outside          → O

Rules:
- Output exactly one tag per input token, one per line.
- Preserve token order strictly.
- Use B- for the first token of an entity span.
- Use I- for continuation tokens of the same entity.
- Output O for non-entity tokens.
- Return ONLY the labels, nothing else."""


def build_user_message(tokens: list[str]) -> str:
    sentence = " ".join(tokens)
    token_list = "\n".join(tokens)

    return (
        f"Sentence:\n{sentence}\n\n"
        "Assign one IOB2 label to each token below.\n"
        "Return ONLY the labels in order, one per line.\n\n"
        f"Tokens:\n{token_list}"
    )


def parse_ground_truth(label_str):
    """Parse ground truth labels, merging split multi-word labels."""
    parts = str(label_str).split(" ")
    labels = []
    j = 0
    while j < len(parts):
        token = parts[j]
        if token in GT_LABEL_MAP:
            # next part is "Condition" or "Procedure" — skip it
            labels.append(GT_LABEL_MAP[token])
            j += 2
        else:
            labels.append(token)
            j += 1
    return labels


def parse_response(raw_text, num_tokens):
    """Parse model output into a fixed-length label list."""
    if raw_text is None:
        return ["O"] * num_tokens
    lines = [l.strip() for l in raw_text.strip().split("\n") if l.strip()]
    parsed = []
    for line in lines:
        if line in VALID_SET:
            parsed.append(line)
        else:
            parsed.append(VALID_LOWER.get(line.lower(), "O"))
    if len(parsed) < num_tokens:
        parsed += ["O"] * (num_tokens - len(parsed))
    return parsed[:num_tokens]


def build_headers():
    return {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
        "HTTP-Referer":  SITE_URL,
        "X-Title":       SITE_NAME,
    }


async def call_model_async(session, model, user_message, headers, num_tokens):
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        "max_tokens": max(num_tokens * 8, 128),
        "temperature": 0,
        "reasoning": {
          "effort": "none"
        },
    }
    start = time.monotonic()
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            async with session.post(
                OPENROUTER_URL, headers=headers, json=payload,
                timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)
            ) as resp:
                if resp.status == 429:
                    wait = float(resp.headers.get("Retry-After", BASE_BACKOFF * attempt))
                    await asyncio.sleep(wait)
                    continue
                if resp.status >= 500:
                    await asyncio.sleep(BASE_BACKOFF * attempt)
                    continue
                if resp.status != 200:
                    text = await resp.text()
                    return None, f"http_error:{resp.status}:{text[:200]}", time.monotonic() - start
                data    = await resp.json()
                content = data["choices"][0]["message"]["content"]
                return content, None, time.monotonic() - start
        except Exception as e:
            if attempt == MAX_RETRIES:
                return None, f"request_error:{e}", time.monotonic() - start
            await asyncio.sleep(BASE_BACKOFF * attempt)
    return None, "max_retries_exceeded", time.monotonic() - start


def checkpoint_path(out_dir, model):
    return os.path.join(out_dir, f"checkpoint_{model.replace('/', '__')}.csv")

def load_checkpoint(out_dir, model):
    path = checkpoint_path(out_dir, model)
    return pd.read_csv(path) if os.path.exists(path) else None

def save_checkpoint(out_dir, model, rows):
    pd.DataFrame(rows).to_csv(checkpoint_path(out_dir, model), index=False)


async def run_model_async(model, df, headers, out_dir):
    existing  = load_checkpoint(out_dir, model)
    rows_done = existing.to_dict("records") if existing is not None else []
    start_idx = len(rows_done)

    if start_idx >= len(df):
        print("Already complete — loaded from checkpoint.")
        return rows_done

    sem     = asyncio.Semaphore(CONCURRENCY)
    indices = list(range(start_idx, len(df)))
    batches = [indices[i:i + BATCH_SIZE] for i in range(0, len(indices), BATCH_SIZE)]
    results = []

    async def process_row(session, row_idx):           # FIX: renamed i → row_idx
        async with sem:
            row         = df.iloc[row_idx]
            tokens      = str(row["text"]).split()
            true_labels = parse_ground_truth(row["labels"])

            min_len     = min(len(tokens), len(true_labels))
            tokens      = tokens[:min_len]
            true_labels = true_labels[:min_len]

            user_msg = build_user_message(tokens)
            raw, err, latency = await call_model_async(session, model, user_msg, headers, min_len)
            pred_labels = parse_response(raw, min_len)

            # enforce same length after parse
            min_len = min(len(true_labels), len(pred_labels))
            true_labels = true_labels[:min_len]
            pred_labels = pred_labels[:min_len]

            print(f"[{row_idx}] tokens={len(tokens)} pred_lines={len(raw.splitlines()) if raw else 0} err={err}")

            return {
                "_idx":          row_idx,
                "text":          row["text"],
                "true_labels":   " ".join(true_labels),
                "pred_labels":   " ".join(pred_labels),
                "primary_class": row.get("primary_class", ""),
                "latency":       latency,
                "error":         err,
            }

    async with aiohttp.ClientSession() as session:
        for b_num, batch in enumerate(batches):
            print(f"\nBatch {b_num + 1}/{len(batches)} — rows {batch[0]}..{batch[-1]}")
            tasks         = [process_row(session, i) for i in batch]
            batch_results = await asyncio.gather(*tasks)
            results.extend(batch_results)

            all_rows = rows_done + sorted(results, key=lambda r: r["_idx"])
            save_checkpoint(out_dir, model, all_rows)

            if b_num < len(batches) - 1:
                print(f"Waiting {BATCH_DELAY}s…")
                await asyncio.sleep(BATCH_DELAY)

    results = sorted(results, key=lambda r: r["_idx"])
    for r in results:
        del r["_idx"]
    all_rows = rows_done + results
    save_checkpoint(out_dir, model, all_rows)
    return all_rows


def run_model_on_dataset(model, df, headers, out_dir):
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(run_model_async(model, df, headers, out_dir))


def compute_metrics(rows):
    y_true, y_pred         = [], []
    all_true_flat, all_pred_flat = [], []
    latencies              = []

    for r in rows:
        if pd.notna(r.get("true_labels")) and pd.notna(r.get("pred_labels")):
            t = str(r["true_labels"]).split()
            p = str(r["pred_labels"]).split()
            min_len = min(len(t), len(p))
            y_true.append(t[:min_len])
            y_pred.append(p[:min_len])
            all_true_flat.extend(t[:min_len])
            all_pred_flat.extend(p[:min_len])
        if r.get("latency") is not None:
            latencies.append(r["latency"])

    correct        = sum(t == p for t, p in zip(all_true_flat, all_pred_flat))
    token_accuracy = correct / len(all_true_flat) if all_true_flat else 0

    return {
        "token_accuracy": token_accuracy,
        "f1":             f1_score(y_true, y_pred, average="weighted"),
        "precision":      precision_score(y_true, y_pred, average="weighted"),
        "recall":         recall_score(y_true, y_pred, average="weighted"),
        "total":          len(rows),
        "avg_latency":    sum(latencies) / len(latencies) if latencies else None,
        "report":         classification_report(y_true, y_pred, digits=4),
    }


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(CSV_PATH)
    if not {"text", "labels"}.issubset(df.columns):
        raise ValueError("CSV must contain 'text' and 'labels' columns")

    if NUM_SAMPLES is not None:
        df = df.head(NUM_SAMPLES).reset_index(drop=True)

    print(f"\nDataset: {len(df)} sentences")
    from collections import Counter
    all_labels = [lbl for row in df["labels"] for lbl in str(row).split()]
    print("Label distribution:")
    for lbl, cnt in sorted(Counter(all_labels).items()):
        print(f"  {lbl:<30} {cnt}")

    headers = build_headers()
    rows    = run_model_on_dataset(MODEL, df, headers, OUTPUT_DIR)
    metrics = compute_metrics(rows)

    pd.DataFrame(rows).to_csv(
        os.path.join(OUTPUT_DIR, "predictions.csv"), index=False, encoding="utf-8-sig")

    results_df = pd.DataFrame([{
        "model":              MODEL,
        "token_accuracy":     metrics["token_accuracy"],
        "f1_weighted":        metrics["f1"],
        "precision_weighted": metrics["precision"],
        "recall_weighted":    metrics["recall"],
        "total_sentences":    metrics["total"],
        "avg_latency_s":      metrics["avg_latency"],
    }])
    results_df.to_csv(
        os.path.join(OUTPUT_DIR, "benchmark_results.csv"), index=False, encoding="utf-8-sig")

    print("\n── Overall Results ──")
    print(results_df.to_string(index=False))
    print("\n── seqeval Span-Level Report (per entity type) ──")
    print(metrics["report"])


if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Dataset: 2000 sentences
Label distribution:
  B-Age                          349
  B-Dosage                       483
  B-Health                       493
  B-Medical                      177
  B-Medicine                     1164
  B-Specialist                   423
  B-Symptom                      2240
  Condition                      1170
  I-Age                          317
  I-Dosage                       1965
  I-Health                       677
  I-Medical                      257
  I-Medicine                     1640
  I-Specialist                   438
  I-Symptom                      6649
  O                              53497
  Procedure                      434

Batch 1/127 — rows 107..121
[111] tokens=15 pred_lines=15 err=None
[107] tokens=27 pred_lines=26 err=None
[109] tokens=20 pred_lines=20 err=None
[108] tokens=37 pred_lines=38 err=None
[110